# Ansatz Design Comparison

UCCSD-style double excitation vs hardware-efficient ansatz on H2: depth, CNOTs, params, energy. Full prose on disk.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.chemistry.ansatz import hardware_efficient_ansatz, uccsd_singles_circuit
from lib.utils.statevector import statevector
device = LocalSimulator()
np.random.seed(0)

In [ ]:
# --- Chemistry kit (auto-provided; real STO-3G H2 data + exact helpers) ---
# H2 Jordan-Wigner qubit Hamiltonian at R = 0.75 Angstrom (STO-3G, big-endian:
# character k of each Pauli string acts on qubit k, qubit 0 = leftmost = MSB).
# Coefficients are REAL precomputed values (PennyLane differentiable Hartree-Fock).
H2_TERMS = [
    ("IIII", -0.1097305),
    ("IIIZ", -0.21886309),
    ("IIZI", -0.21886309),
    ("IIZZ", 0.17395379),
    ("IZII", 0.16988453),
    ("IZIZ", 0.12005143),
    ("IZZI", 0.16549432),
    ("XXYY", -0.04544288),
    ("XYYX", 0.04544288),
    ("YXXY", 0.04544288),
    ("YYXX", -0.04544288),
    ("ZIII", 0.16988453),
    ("ZIIZ", 0.16549432),
    ("ZIZI", 0.12005143),
    ("ZZII", 0.16821199),
]
H2_FCI = -1.137117   # exact ground-state energy (lowest eigenvalue), Hartree
H2_HF = -1.116151    # Hartree-Fock energy <1100|H|1100>, Hartree
# Z2 symmetry-tapered single-qubit form: H = c0*I + cz*Z + cx*X (ground = c0 - hypot(cz, cx))
H2_C0, H2_CZ, H2_CX = -0.338656, 0.777495, 0.181772

_PAULI = {
    "I": np.array([[1, 0], [0, 1]], dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def pauli_matrix(pauli_string):
    """Dense matrix of a big-endian Pauli string (qubit 0 = leftmost factor)."""
    m = np.array([[1.0 + 0j]])
    for ch in pauli_string:
        m = np.kron(m, _PAULI[ch])
    return m

def hamiltonian_matrix(terms):
    """Dense Hamiltonian sum(coeff * Pauli) from a list of (pauli_string, coeff)."""
    n = len(terms[0][0])
    h = np.zeros((2 ** n, 2 ** n), dtype=complex)
    for s, c in terms:
        h += c * pauli_matrix(s)
    return h

def hamiltonian_energy(state_vector, terms):
    """Expectation <psi|H|psi> for a qcsim state vector (real, in Hartree)."""
    h = hamiltonian_matrix(terms)
    return float(np.real(np.vdot(state_vector, h @ state_vector)))

def h2_double_excitation(theta):
    """HF |1100> plus the single Givens double excitation |1100> <-> |0011>.
    At the optimal theta this ansatz reaches the exact H2 ground state."""
    c = Circuit()
    c.x(0); c.x(1)
    c.cnot(2, 3); c.cnot(0, 2); c.h(0); c.h(3); c.cnot(0, 1); c.cnot(2, 3)
    c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(0, 3); c.h(3); c.cnot(3, 1); c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(2, 1); c.cnot(2, 0); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(3, 1); c.h(3); c.cnot(0, 3); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(0, 1); c.cnot(2, 0); c.h(0); c.h(3); c.cnot(0, 2); c.cnot(2, 3)
    return c
# --- end chemistry kit ---

## 1. What the kit hands us

H2_TERMS (big-endian JW), H2_FCI, H2_HF, hamiltonian_energy, hamiltonian_matrix, h2_double_excitation. A molecule is an operator; its matrix has a lowest eigenvalue we chase by minimizing an expectation value.

In [ ]:
print(f"H2_HF  = {H2_HF:+.6f} Ha")
print(f"H2_FCI = {H2_FCI:+.6f} Ha")
print(f"HF-FCI = {H2_HF - H2_FCI:+.6f} Ha")
H_dense = hamiltonian_matrix(H2_TERMS)
lowest_eig = float(np.linalg.eigvalsh(H_dense)[0])
assert abs(lowest_eig - H2_FCI) < 1e-6, "H2_FCI should equal the lowest eigenvalue of H2_TERMS"

## 2. Chemistry-inspired double excitation

|psi(theta)> = cos(theta/2)|1100> - sin(theta/2)|0011>. One angle reaches FCI, so a 1-D grid search suffices.

In [ ]:
thetas = np.linspace(-np.pi, np.pi, 400)
double_energies = np.array([hamiltonian_energy(statevector(h2_double_excitation(t)), H2_TERMS) for t in thetas])
best_idx = int(np.argmin(double_energies))
best_theta = thetas[best_idx]
best_double_energy = double_energies[best_idx]
print(f"theta={best_theta:+.4f}  E={best_double_energy:+.6f}  FCI={H2_FCI:+.6f}  err={abs(best_double_energy-H2_FCI):.2e}")
assert abs(best_double_energy - H2_FCI) < 1e-3, "the double-excitation ansatz must reach the FCI energy"

In [ ]:
double_circ = h2_double_excitation(best_theta)
double_depth = double_circ.depth
double_cnots = sum(1 for ins in double_circ.instructions if ins.operator.name == "CNot")
double_nparams = 1
print(f"double-exc: depth={double_depth} CNOTs={double_cnots} params={double_nparams} E={best_double_energy:+.6f}")

## 3. Hardware-efficient ansatz

Ry/Rz + linear CNOT chain, params (n_layers, n_qubits, 2). No chemical prior; optimized with a short seeded VQE (few restarts, bounded parameter-shift gradient steps).

In [ ]:
N_QUBITS = 4
N_LAYERS = 1
DIM = N_LAYERS * N_QUBITS * 2
def hea_energy(flat_params):
    params = flat_params.reshape(N_LAYERS, N_QUBITS, 2)
    sv = statevector(hardware_efficient_ansatz(N_QUBITS, N_LAYERS, params))
    return hamiltonian_energy(sv, H2_TERMS)
def param_shift_grad(flat_params):
    grad = np.zeros_like(flat_params)
    for i in range(len(flat_params)):
        plus = flat_params.copy();  plus[i]  += np.pi / 2
        minus = flat_params.copy(); minus[i] -= np.pi / 2
        grad[i] = 0.5 * (hea_energy(plus) - hea_energy(minus))
    return grad

In [ ]:
np.random.seed(0)
N_RESTARTS, N_STEPS, LEARNING_RATE = 6, 8, 0.2
best_he_energy = np.inf
best_he_params = None
for restart in range(N_RESTARTS):
    x = np.random.uniform(-np.pi, np.pi, DIM)
    for _ in range(N_STEPS):
        x = x - LEARNING_RATE * param_shift_grad(x)
    e = hea_energy(x)
    if e < best_he_energy:
        best_he_energy = e
        best_he_params = x.copy()
print(f"best HEA E={best_he_energy:+.6f}  FCI={H2_FCI:+.6f}  gap={best_he_energy-H2_FCI:+.4f}  beatHF={best_he_energy < H2_HF}")
assert best_he_energy > H2_FCI + 0.1, "the hardware-efficient ansatz should remain stuck above FCI in this budget"

In [ ]:
he_circ = hardware_efficient_ansatz(N_QUBITS, N_LAYERS, best_he_params.reshape(N_LAYERS, N_QUBITS, 2))
he_depth = he_circ.depth
he_cnots = sum(1 for ins in he_circ.instructions if ins.operator.name == "CNot")
he_nparams = DIM
print(f"HEA: depth={he_depth} CNOTs={he_cnots} params={he_nparams} E={best_he_energy:+.6f} gap={best_he_energy-H2_FCI:+.4f}")

## 4. Side-by-side comparison

In [ ]:
rows = [("Double excitation", double_depth, double_cnots, double_nparams, best_double_energy),
        ("Hardware-efficient", he_depth, he_cnots, he_nparams, best_he_energy)]
for name, d, c, p, e in rows:
    print(f"{name:<20} depth={d:>3} CNOTs={c:>3} params={p:>2} E={e:+.6f} err={e-H2_FCI:+.4f}")
print(f"{'exact (FCI)':<20} {'-':>9} {'-':>8} {'-':>8} E={H2_FCI:+.6f}")
print("\nThe doubles ansatz reaches the floor with ONE parameter; the HEA is stranded above it.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(thetas, double_energies, color="#2563eb", lw=2, label="double excitation")
ax1.axhline(H2_FCI, color="#16a34a", ls="--", lw=1.5, label="FCI (exact)")
ax1.scatter([best_theta], [best_double_energy], color="#16a34a", zorder=5, s=40)
ax1.set_xlabel("theta (rad)"); ax1.set_ylabel("energy (Ha)")
ax1.set_title("Double excitation landscape"); ax1.legend(fontsize=8)
bars = ax2.bar(["double", "HEA"], [best_double_energy, best_he_energy], color=["#2563eb", "#dc2626"], width=0.55)
ax2.axhline(H2_FCI, color="#16a34a", ls="--", lw=1.5, label="FCI")
ax2.axhline(H2_HF, color="#6b7280", ls=":", lw=1.5, label="HF")
ax2.set_ylabel("best energy (Ha)"); ax2.set_title("Best energy reached"); ax2.legend(fontsize=8)
fig.tight_layout(); plt.show()

## 5. Why ansatz design matters

Accuracy comes from structure, not parameter count. The HEA is under-structured; adding layers invites barren plateaus. The NISQ trade-off: UCCSD (accurate, deep) vs HEA (shallow, hard to train deep) vs ADAPT-VQE (compact, grows operator-by-operator). Full table on disk.

## Exercises

Three exercises on ansatz design. Each has tiered hints -- expand only what you
need -- and a check cell that confirms the property once your attempt fills in
the named variable. Local simulator only; seed your restarts (np.random.seed)
so a run is reproducible.

### Exercise 1 — Deeper HEA: does adding layers buy accuracy?

Rebuild the hardware-efficient ansatz at `n_layers = 1, 2, 3`, optimize each
with the same restart + gradient-descent loop from Section 3, and record how
circuit cost and energy move together.

Define `hea_layer_rows`: one entry per layer count, each a
`(n_layers, depth, cnots, best_energy)` tuple.

<details><summary>Hint 1 — nudge</summary>

Section 3 built and optimized the ansatz at a single layer count. The parameter
tensor has shape `(n_layers, n_qubits, 2)`, so the flat dimension you optimize
grows with `n_layers`. What else grows in the circuit as you stack layers, and
does that extra cost actually pull the energy down?

</details>
<details><summary>Hint 2 — approach</summary>

Loop over `n_layers in (1, 2, 3)`. For each, the flat parameter count is
`n_layers * N_QUBITS * 2`; you will need energy and parameter-shift-gradient
helpers that take `n_layers` as an argument, since the notebook's `hea_energy`
and `param_shift_grad` are hard-wired to `N_LAYERS`. Read `circuit.depth` and
count the CNOT instructions exactly as Section 3 does, then append a
`(n_layers, depth, cnots, best_energy)` tuple.

</details>

In [ ]:
# Exercise 1: sweep the hardware-efficient ansatz over n_layers = 1, 2, 3.
# Define: hea_layer_rows -- one (n_layers, depth, cnots, best_energy) entry per
#         layer count, each optimized with the Section 3 restart + GD loop.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(hea_layer_rows) == 3, "record one row for each of n_layers = 1, 2, 3"
    _rows = sorted(hea_layer_rows)
    assert [r[0] for r in _rows] == [1, 2, 3], "cover layer counts 1, 2, and 3"
    _depths = [r[1] for r in _rows]
    _cnots = [r[2] for r in _rows]
    _energies = [r[3] for r in _rows]
    assert _cnots[0] < _cnots[1] < _cnots[2], "each extra layer adds another CNOT chain"
    assert _depths[0] < _depths[1] < _depths[2], "each extra layer deepens the circuit"
    assert all(e >= H2_FCI - 1e-6 for e in _energies), (
        "a variational energy can never fall below the exact FCI floor"
    )

### Exercise 2 — Optimization budget: how many steps until it pays off?

Hold the ansatz at `N_LAYERS = 3` and vary only the number of gradient-descent
steps. Show how the best energy you can reach responds to the optimization
budget.

Define `step_budgets = [2, 8, 32]` and `energy_per_budget`: the best energy
reached at each budget, in the same order, with the layer count fixed at 3.

<details><summary>Hint 1 — nudge</summary>

The Section 3 loop took a fixed number of gradient steps. Here the only thing
that changes between runs is that step count; the ansatz, learning rate, and
restart strategy stay put. What should more descent steps do to the energy you
land on, and does each extra step keep paying off or taper?

</details>
<details><summary>Hint 2 — approach</summary>

Loop over `step_budgets`. For each budget, run the same restart +
parameter-shift gradient descent from Section 3 (fixed `N_LAYERS = 3`, same
learning rate) but cap the inner loop at that many steps, and append the best
energy across restarts. Re-seed `np.random` before each budget so every budget
starts from the same restarts and only the step count differs.

</details>

In [ ]:
# Exercise 2: fix N_LAYERS = 3 and sweep the gradient-step budget.
# Define: step_budgets = [2, 8, 32], and energy_per_budget -- the best energy
#         reached at each budget (same order), everything else held fixed.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert list(step_budgets) == [2, 8, 32], "use the three step budgets from the prompt"
    assert len(energy_per_budget) == 3, "record one best energy per budget"
    assert all(e >= H2_FCI - 1e-6 for e in energy_per_budget), (
        "every VQE energy sits on or above the exact FCI floor"
    )
    assert energy_per_budget[-1] <= energy_per_budget[0] + 0.05, (
        "the largest budget should not end up worse than the smallest"
    )

### Exercise 3 — Singles-only: where UCCSD stops short

Build the singles-only UCCSD ansatz `uccsd_singles_circuit(4, 2, params)`,
optimize its excitation amplitudes against the H2 Hamiltonian, and confirm the
best energy it reaches still sits above FCI.

Define `singles_best_energy`: the lowest energy you reach by optimizing the
parameters of the singles-only ansatz.

<details><summary>Hint 1 — nudge</summary>

The double excitation in Section 2 was what let the state mix `|1100>` with
`|0011>` and reach the floor. Single excitations move one electron at a time
from an occupied to a virtual orbital. Ask whether any sequence of such moves
can assemble the correlated superposition the true ground state needs.

</details>
<details><summary>Hint 2 — approach</summary>

The ansatz takes `n_electrons * (n_qubits - n_electrons)` amplitudes (four, for
4 qubits / 2 electrons). Wrap `statevector(uccsd_singles_circuit(4, 2, params))`
in an energy function like Section 3's `hea_energy`, optimize it with the same
parameter-shift gradient descent over a few random restarts, and keep the
lowest energy. Compare it to `H2_FCI`.

</details>

In [ ]:
# Exercise 3: optimize the singles-only UCCSD ansatz and see where it plateaus.
# Define: singles_best_energy -- the lowest energy reached by optimizing the
#         parameters of uccsd_singles_circuit(4, 2, params) against H2_TERMS.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert singles_best_energy >= H2_FCI - 1e-6, (
        "a variational energy can never fall below the exact FCI floor"
    )
    assert singles_best_energy > H2_FCI + 0.01, (
        "single excitations alone cannot reach the correlated ground state -- "
        "the best singles energy should stay above FCI"
    )

## Summary

- Doubles ansatz hit H2_FCI (asserted 1e-3) with one parameter.
- HEA stayed stuck above FCI (asserted > H2_FCI + 0.1) in the short budget.
- Lesson: accuracy / depth / trainability trade-off (UCCSD vs HEA vs ADAPT-VQE).

**Next:** [`06-active-space.ipynb`](06-active-space.ipynb).